# N22 · Weight Sync 的真正机制：handle tuple 不是数据

> 配套 `labs/l32.5_ipc_weight_sync/`。
>
> 读完后你应能：
> 1. 解释 verl `update_weights_from_tensor` 为什么序列化 bytes 远小于 tensor 数据
> 2. 区分 `from_disk` / `from_distributed` / `from_tensor` 三种接口的取舍
> 3. 估算 70B 模型在 co-locate 下 weight sync 的耗时构成
>
> 参考：[RL 系统深思 · 权重更新机制](../github_repo/Awesome-ML-SYS-Tutorial/rlhf/sys-design/readme-1.md)


## 1. 序列化 ≠ 复制

verl 的 `MultiprocessingSerializer.serialize(tensor)` 返回的不是 tensor data 的 pickle bytes，而是一个 14 元组：

```python
(rebuild_cuda_tensor, (
    type(tensor),       # tensor 类型
    tensor.size(),      # shape
    tensor.stride(),    # stride
    tensor_offset,      # 在 storage 里的偏移
    type(storage),      # storage 类型
    tensor.dtype,
    device,
    handle,             # CUDA IPC handle
    storage_size_bytes,
    storage_offset_bytes,
    tensor.requires_grad,
    ref_counter_handle,
    ref_counter_offset,
    event_handle,
    event_sync_required,
))
```

关键是 `handle`——CUDA IPC handle 是个 64-byte 的描述符，**接收端用它就能映射到同一块 GPU 物理内存**，没有数据复制。


In [ ]:
# CPU 模拟：把 tensor 注册到 pool 拿 uuid handle
import pickle, uuid, torch

class IPCStoragePool:
    def __init__(self):
        self._t = {}
    def put(self, t):
        h = uuid.uuid4().hex
        self._t[h] = t
        return h
    def get(self, h):
        return self._t[h]

pool = IPCStoragePool()
tensor = torch.randn(1024, 1024)  # 4 MB at fp32

def serialize(t):
    h = pool.put(t)
    meta = {"handle": h, "shape": list(t.shape), "dtype": str(t.dtype)}
    return pickle.dumps(meta)

blob = serialize(tensor)
print(f"tensor data size: {tensor.numel() * tensor.element_size():,} bytes")
print(f"serialized blob:  {len(blob):,} bytes  (-> {len(blob) * 100 / (tensor.numel() * tensor.element_size()):.4f}% of data)")

In [ ]:
# 接收端反序列化得到与原 tensor share storage 的 tensor
def deserialize(blob):
    meta = pickle.loads(blob)
    return pool.get(meta["handle"])

rebuilt = deserialize(blob)
print(f"data_ptr equal: {rebuilt.data_ptr() == tensor.data_ptr()}  (即 share storage)")
print(f"values equal:   {torch.equal(rebuilt, tensor)}")
print("\n语义：rebuilt 与 tensor 指向同一块显存。SGLang 拿到后直接 load_weights，省掉跨进程数据搬运。")

## 2. 三种 update_weights 接口的取舍

| 接口 | 动作 | 适用 placement | 耗时构成 | 扩缩容 |
|---|---|---|---|---|
| `update_weights_from_disk` | 写盘 + Engine 读盘 | 二者都行 | 决定于盘 IO；checkpoint 一并完成 | ★★★ 简单：DP router 加新 engine 即可 |
| `update_weights_from_distributed` | NCCL/IB broadcast 跨组传数据 | 仅 disaggregate | NCCL bandwidth × tensor 总字节 | ★ 复杂：需要重建通讯组 |
| `update_weights_from_tensor` | gather IPC handle，引用共享 | 仅 co-locate | gather + serialize ≈ 几百 ms | ★ 不支持（同进程） |

生产级 RL 框架的选择：
- **verl** 默认 `from_tensor` (co-locate)
- **slime** 同时支持 co-locate 与 disaggregate；co-locate 走 from_tensor + 桶状更新避免 OOM
- **AReaL** 走 `from_disk`，因为它本质是 disaggregate 异步 RL，扩缩容是核心特性


In [ ]:
# 估算 70B 模型 weight sync 时间
# Llama-2 70B: ~140 GB at fp16

GB = 1024 ** 3
weight_size_bytes = 140 * GB

# Disaggregate via NCCL @ 50 GB/s per GPU
nccl_bandwidth = 50 * GB
from_distributed_seconds = weight_size_bytes / nccl_bandwidth
print(f"from_distributed (NCCL 50GB/s):  {from_distributed_seconds:.2f} s")

# Disk @ 5 GB/s
disk_bandwidth = 5 * GB
from_disk_seconds = (weight_size_bytes / disk_bandwidth) * 2  # write + read
print(f"from_disk     (NVMe 5GB/s, 2x):  {from_disk_seconds:.2f} s")

# IPC handle: 0 数据移动；但每参数有几百 us 的 gather + serialize
num_params = 700  # rough for 70B
from_tensor_seconds = num_params * 0.5e-3  # 0.5 ms per param
print(f"from_tensor   (IPC handles only): {from_tensor_seconds * 1000:.1f} ms")

## 3. 自检 / 面试题

1. `LocalSerializedTensor.values` 是 `list[bytes]`，每个 bytes 都是一个 rank 的 handle blob。为什么需要 list 而不是单一 blob？（答：每个 SGLang TP rank 拿到自己 rank 对应的那个 handle 重建 tensor，不同 rank 的物理内存切片不同）
2. flush_cache 为什么只在最后一个 tensor 上触发？（答：每次 update_weights_from_tensor 涉及多个参数，整个调用结束才能安全清掉 KV cache，否则中途的请求拿到的是新旧权重混合的输出）
3. Co-locate 下为什么 SGLang 的 mem_static_fraction 经常压不到很高？（答：rollout engine 必须给训练后端 offload 的优化器和梯度状态留空间，offload 不完美时 fraction 提不上去）
